<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "678"
iso_country = "Sao Tome and Principe"
year = 2021
import_portfolio = "h3_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h3_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 520*1e6

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Sao Tome and Principe': 

Number of trade records present in dataset: 6098
Number of partner countries present in dataset: 52
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$120.4m
Total weight within the dataset: 187.1 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,678,Sao Tome and Principe,724,Spain,Import,Sea,950640,950640 | Table-tennis article,9506,"9506 | Gymnastics, athletic",95,"95 | Toys, games and spor",3844,Other articles and equipment for sports or out...,Other Manufacturing,42,OMF - Manufactures n.e.c.,8187.482,0,8187.482,kg,957.000,957.0,944.016675,6263.0,3891.646573,3673.779257,['Container'],['Container'],17.199571,17.167257,17.170531,17.170531,17.184960,17.184960,18.007631,32.114422,19.718410,18.462977,18.515694,18.949815,17.625396,31.368026,20.842318,19.146254,18.690938,19.559952,17.290157,34.372342,25.911769,22.681070,20.958744,20.39751,16.961128,36.465694,29.982197,27.925602,22.420368,21.874544,16.638311,37.141581,35.674654,36.096203,24.158894,22.984126,16.309023,36.364134,34.561803,37.728023,28.447354,23.411098,15.998810,32.230743,33.762400,35.318021,26.033439,23.688168,15.682491,29.626780,30.050240,36.466335,30.347497,23.851295,3844
1,1,678,Sao Tome and Principe,156,China,Import,Sea,950651,950651 | Lawn-tennis racquets,9506,"9506 | Gymnastics, athletic",95,"95 | Toys, games and spor",3844,Other articles and equipment for sports or out...,Other Manufacturing,42,OMF - Manufactures n.e.c.,2.578,0,2.578,u,0.111,1.0,0.297243,19435.0,12076.345385,3.589614,['Container'],['Container'],0.016806,0.016774,0.016777,0.016777,0.016791,0.016791,0.017595,0.031379,0.019267,0.018040,0.018092,0.018516,0.017222,0.030649,0.020365,0.018708,0.018263,0.019112,0.016894,0.033585,0.025318,0.022161,0.020479,0.01993,0.016573,0.035630,0.029295,0.027286,0.021907,0.021373,0.016257,0.036291,0.034857,0.035269,0.023605,0.022458,0.015935,0.035531,0.033770,0.036864,0.027796,0.022875,0.015632,0.031492,0.032989,0.034509,0.025437,0.023145,0.015323,0.028948,0.029362,0.035631,0.029652,0.023305,3844


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $55.3m (45.9%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
22 | Beverages, spirits a  11145139.8  25490578.2      9.3     13.6
10 | Cereals                7986390.5  13983020.8      6.6      7.5
11 | Products of the mill   5888587.0  13505721.2      4.9      7.2
02 | Meat and edible meat   4087290.1   2171469.3      3.4      1.2
19 | Preparations of cere   3749950.8   1733455.6      3.1      0.9
04 | Dairy produce; birds   3663327.4   2458452.9      3.0      1.3
15 | Animal, vegetable or   3651200.0   3393500.2      3.0      1.8
23 | Food industries, res   3104071.1   2296702.2      2.6      1.2
20 | Preparations of vege   2566791.5   2027510.8      2.1      1.1
07 | Vegetables and certa   2504118.8   4082217.5      2.1      2.2
17 | Sugars and sugar con   2399462.0   4434714.3      2.0      2.4
16 | Meat, fish,

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $55.4m (46.0%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Agriculture       28582730.1  40212619.0     23.7     21.5
Food & Beverages  26712741.5  36976113.2     22.2     19.8
Fishing              95656.4     22617.7      0.1      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $43.6m (36.2%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     17082458.0  18840371.4     14.2     10.1
PCR - Processed rice            7933602.9  13780668.2      6.6      7.4
OMT - Meat products n.e.c.      5830928.7   2577761.8      4.8      1.4
MIL - Dairy products            3845807.1   2517142.8      3.2      1.3
VOL - Vegetable oils and fats   3650307.6   3393070.4      3.0      1.8
V_F - Vegetables, fruit, nuts   2982489.2   4540842.2      2.5      2.4
SGR - Sugar                     1914599.1   4201025.8      1.6      2.2
CMT - Bovine meat prods          169143.1     40359.1      0.1      0.0
OCR - Crops n.e.c.               121303.3     48667.5      0.1      0.0
OSD - Oil seeds                   47252.8     40985.3      0.0      0.0
GRO - Cereal grains n.e.c.        46614.4    191629.8      0.0      0.1

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 45.9% 
GTAP 36.2% 
EORA 46.0%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46410/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,678,Sao Tome and Principe,724,Spain,Import,Sea,950640,950640 | Table-tennis article,9506,"9506 | Gymnastics, athletic",95,"95 | Toys, games and spor",3844,Other articles and equipment for sports or out...,Other Manufacturing,42,OMF - Manufactures n.e.c.,8187.482,0,8187.482,kg,957.000,957.0,944.016675,6263.0,3891.646573,3673.779257,['Container'],['Container'],17.199571,17.167257,17.170531,17.170531,17.184960,17.184960,18.007631,32.114422,19.718410,18.462977,18.515694,18.949815,17.625396,31.368026,20.842318,19.146254,18.690938,19.559952,17.290157,34.372342,25.911769,22.681070,20.958744,20.39751,16.961128,36.465694,29.982197,27.925602,22.420368,21.874544,16.638311,37.141581,35.674654,36.096203,24.158894,22.984126,16.309023,36.364134,34.561803,37.728023,28.447354,23.411098,15.998810,32.230743,33.762400,35.318021,26.033439,23.688168,15.682491,29.626780,30.050240,36.466335,30.347497,23.851295,3844,NaN
1,1,678,Sao Tome and Principe,156,China,Import,Sea,950651,950651 | Lawn-tennis racquets,9506,"9506 | Gymnastics, athletic",95,"95 | Toys, games and spor",3844,Other articles and equipment for sports or out...,Other Manufacturing,42,OMF - Manufactures n.e.c.,2.578,0,2.578,u,0.111,1.0,0.297243,19435.0,12076.345385,3.589614,['Container'],['Container'],0.016806,0.016774,0.016777,0.016777,0.016791,0.016791,0.017595,0.031379,0.019267,0.018040,0.018092,0.018516,0.017222,0.030649,0.020365,0.018708,0.018263,0.019112,0.016894,0.033585,0.025318,0.022161,0.020479,0.01993,0.016573,0.035630,0.029295,0.027286,0.021907,0.021373,0.016257,0.036291,0.034857,0.035269,0.023605,0.022458,0.015935,0.035531,0.033770,0.036864,0.027796,0.022875,0.015632,0.031492,0.032989,0.034509,0.025437,0.023145,0.015323,0.028948,0.029362,0.035631,0.029652,0.023305,3844,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $36.4m of $120.4m (30.2%)
Volume: 46.5 of 187.1 million kilograms (24.9%)
Value-to-Weight: 0.78 from 0.64 US$ per kg (+21.7%)
Average distance to market (volume-weighted): 5931.5 from 3793.7 nm (+56.4%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    209  19181858.2  29078169.2     15.9     15.5
1        Meat and Prep     38   3886353.8   2129249.8      3.2      1.1
2           Vegetables    101   3737584.8   5106645.3      3.1      2.7
3                Dairy     47   3656443.7   2456407.1      3.0      1.3
4        Fats and Oils     29   2971210.2   2598820.5      2.5      1.4
5      Sugar and Honey     34   2401274.2   4435233.8      2.0      2.4
6               Fruits     62    536124.3    506135.5      0.4      0.3
7  Unprocessed Cereals      7     46614.4    191629.8      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,4637048.2,8154157.6,5050981.0,4743215.4,4756088.0,4868997.9,4296303.1,9480902.5,9057245.5,9216948.8,6163631.7,5881939.3,4056349.2,7562271.1,7670548.6,9249509.4,7672703.2,6102282.2
1,Fats and Oils,4491660.4,7237766.0,4790295.9,4498947.1,4510371.5,4661167.9,4225165.0,8160637.1,7893301.4,8140643.7,5716791.1,5430787.2,4038804.6,6795250.8,6935554.4,8073624.7,6784131.8,5594780.0
2,Fruits,985202.7,1748825.6,1076925.0,1009308.6,1012094.0,1035975.1,911165.1,2024483.2,1941948.2,1967905.1,1317272.5,1254586.1,859340.3,1616160.7,1639296.0,1985059.4,1651198.4,1301928.1
3,Meat and Prep,4753693.9,8473928.9,5204453.7,4873533.8,4887404.6,5002061.5,4392617.4,9801324.9,9413020.1,9525627.1,6375479.5,6066094.1,4140509.9,7818808.0,7930576.1,9621918.1,8007050.2,6294969.0
4,Processed Cereals,92644279.7,164188795.8,101226684.7,94835766.4,95109121.5,97403116.9,85696136.1,190157505.1,182474727.3,184794845.9,124098727.2,117828950.7,80845626.4,151925752.0,154232429.5,186508413.0,155284413.2,122201652.0
5,Sugar and Honey,6573735.5,10567765.2,6932404.9,6624270.2,6630036.0,6808245.0,6197726.0,12502113.5,11643067.4,12199866.4,8201540.6,7980622.5,5915281.6,10154493.5,10309792.9,11902777.0,9799654.4,8280384.2
6,Unprocessed Cereals,275311.8,472567.1,297555.6,279568.0,280413.8,288137.6,256101.3,551309.4,526472.6,535598.1,365964.4,343922.2,242684.5,444105.4,452881.9,537872.9,448908.3,355588.3
7,Vegetables,10852711.0,19053193.3,11814115.5,11097102.9,11125269.7,11391800.7,10059973.1,22109206.1,21139243.1,21500387.2,14395456.2,13745408.3,9501027.7,17681279.1,17935888.0,21606545.2,17951300.7,14264162.5


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.96,0.11,0.03,0.03,0.06,1.42,1.30,1.35,0.51,0.43,0.96,0.99,1.42,0.99,0.56
1,Fats and Oils,0.92,0.10,0.00,0.01,0.06,1.32,1.23,1.32,0.50,0.41,0.93,0.97,1.36,0.92,0.52
2,Fruits,1.42,0.17,0.04,0.05,0.09,2.08,1.92,1.97,0.76,0.64,1.41,1.45,2.10,1.48,0.83
3,Meat and Prep,0.96,0.12,0.03,0.03,0.06,1.39,1.29,1.32,0.51,0.43,0.95,0.98,1.41,0.99,0.55
4,Processed Cereals,3.73,0.45,0.11,0.13,0.25,5.45,5.05,5.17,2.00,1.68,3.71,3.83,5.51,3.88,2.16
5,Sugar and Honey,1.66,0.15,0.02,0.02,0.10,2.63,2.27,2.50,0.83,0.74,1.77,1.83,2.49,1.62,0.98
6,Unprocessed Cereals,4.23,0.48,0.09,0.11,0.28,6.33,5.80,6.00,2.36,1.88,4.32,4.51,6.33,4.42,2.42
7,Vegetables,2.19,0.26,0.07,0.07,0.14,3.22,2.96,3.06,1.16,0.99,2.19,2.26,3.24,2.26,1.27
